## Step 7. 결론 및 요약

### PID 제어 RPM 최적화 결과

**시뮬레이션 설정:**
- 120 타임스텝 (2시간, 1분 간격) 생산 레벨 시뮬레이션
- 4종 소재 (배터리, 플라스틱, 금속, 혼합) x 3종 목표 크기 (30/50/80mm) = 12개 시나리오

**주요 발견:**

1. **PID 제어기 성능:** Anti-windup이 적용된 PID 제어기가 모든 시나리오에서 목표 파쇄 크기로 수렴함을 확인
2. **소재별 차이:**
   - 플라스틱: 낮은 경도로 인해 가장 빠른 수렴 (낮은 응답 지연)
   - 금속: 높은 경도로 인해 가장 느린 수렴, 높은 RPM 필요
   - 혼합: 변동성이 가장 크며 정상상태 오차가 상대적으로 높음
3. **RPM 룩업 테이블:** 소재-크기 조합별 최적 RPM 기준값이 PID 초기값으로 유효함
4. **A축/B축 역할:** A축(주축)이 파쇄 크기에 더 큰 영향을 미치며, B축(보조축)은 미세 조정 역할

**실무 적용 방안:**
- 소재 투입 시 룩업 테이블 기반 초기 RPM 설정 → PID 자동 튜닝으로 목표 수렴
- 소재별 PID 게인을 사전 튜닝하여 최적 응답 확보
- 실시간 파쇄 크기 측정 센서 연동 시 즉시 적용 가능

In [ ]:
# 6-6. Error Convergence Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# 상단: 4종 소재 오차 수렴 (목표 30mm 기준)
for mk in MATERIALS:
    key = f"{mk}_fine"
    r = all_results[key]
    ax1.plot(r['time'], r['error'], linewidth=1.5, alpha=0.8,
             color=MATERIALS[mk]['color'],
             label=f'{MATERIALS[mk]["name_kr"]} (SS err: {r["ss_error"]:.2f}mm)')

ax1.axhline(0, color='black', linestyle='-', alpha=0.3)
ax1.axhline(1.5, color='gray', linestyle=':', alpha=0.4, label='+/- 5% Tolerance')
ax1.axhline(-1.5, color='gray', linestyle=':', alpha=0.4)
ax1.fill_between(range(120), -1.5, 1.5, alpha=0.05, color='green')
ax1.set_ylabel('Error (mm)', fontsize=12)
ax1.set_title('Error Convergence by Material (Target: 30mm Fine)', fontsize=14, fontweight='bold')
ax1.legend(loc='upper right', fontsize=10)
ax1.set_xlim(0, 120)

# 하단: 소재별 절대 오차의 이동평균
window = 10
for mk in MATERIALS:
    key = f"{mk}_fine"
    r = all_results[key]
    abs_error = np.abs(r['error'])
    # 이동평균
    ma = np.convolve(abs_error, np.ones(window)/window, mode='valid')
    time_ma = np.arange(window-1, 120)
    ax2.plot(time_ma, ma, linewidth=2, alpha=0.8,
             color=MATERIALS[mk]['color'],
             label=f'{MATERIALS[mk]["name_kr"]}')

ax2.axhline(0, color='black', linestyle='-', alpha=0.3)
ax2.set_xlabel('Time Step (min)', fontsize=12)
ax2.set_ylabel('Absolute Error - Moving Avg (mm)', fontsize=12)
ax2.set_title(f'Absolute Error Moving Average (Window={window} steps)', fontsize=14, fontweight='bold')
ax2.legend(loc='upper right', fontsize=10)
ax2.set_xlim(0, 120)

plt.tight_layout()
plt.show()

### 6-6. 오차 수렴 플롯
4종 소재(목표 30mm)에 대한 오차의 시간에 따른 수렴 양상을 비교합니다.

In [ ]:
# 6-5. Multi-Scenario Comparison - Settling Time Heatmap
mat_keys = list(MATERIALS.keys())
size_keys = list(TARGET_SIZES.keys())

# 히트맵 데이터 구성
settling_matrix = np.zeros((len(mat_keys), len(size_keys)))
ss_error_matrix = np.zeros((len(mat_keys), len(size_keys)))

for i, mk in enumerate(mat_keys):
    for j, sk in enumerate(size_keys):
        key = f"{mk}_{sk}"
        settling_matrix[i, j] = all_results[key]['settling_time']
        ss_error_matrix[i, j] = all_results[key]['ss_error']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 정착 시간 히트맵
im1 = ax1.imshow(settling_matrix, cmap='YlOrRd', aspect='auto')
ax1.set_xticks(range(len(size_keys)))
ax1.set_xticklabels([TARGET_SIZES[sk]['name_kr'] for sk in size_keys])
ax1.set_yticks(range(len(mat_keys)))
ax1.set_yticklabels([MATERIALS[mk]['name_kr'] for mk in mat_keys])
ax1.set_title('Settling Time (steps)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Target Size', fontsize=11)
ax1.set_ylabel('Material', fontsize=11)

for i in range(len(mat_keys)):
    for j in range(len(size_keys)):
        val = settling_matrix[i, j]
        color = 'white' if val > settling_matrix.max() * 0.6 else 'black'
        ax1.text(j, i, f'{val:.0f}', ha='center', va='center',
                 fontsize=13, fontweight='bold', color=color)

plt.colorbar(im1, ax=ax1, label='Steps')

# 정상상태 오차 히트맵
im2 = ax2.imshow(ss_error_matrix, cmap='Blues', aspect='auto')
ax2.set_xticks(range(len(size_keys)))
ax2.set_xticklabels([TARGET_SIZES[sk]['name_kr'] for sk in size_keys])
ax2.set_yticks(range(len(mat_keys)))
ax2.set_yticklabels([MATERIALS[mk]['name_kr'] for mk in mat_keys])
ax2.set_title('Steady-State Error (mm)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Target Size', fontsize=11)
ax2.set_ylabel('Material', fontsize=11)

for i in range(len(mat_keys)):
    for j in range(len(size_keys)):
        val = ss_error_matrix[i, j]
        color = 'white' if val > ss_error_matrix.max() * 0.6 else 'black'
        ax2.text(j, i, f'{val:.2f}', ha='center', va='center',
                 fontsize=13, fontweight='bold', color=color)

plt.colorbar(im2, ax=ax2, label='mm')

plt.suptitle('Multi-Scenario PID Performance Comparison (4 Materials x 3 Sizes)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 6-5. 다중 시나리오 비교 (정착 시간 히트맵)
12개 시나리오의 정착 시간을 히트맵으로 비교합니다. 정착 시간이 짧을수록 PID 제어 성능이 우수합니다.

In [ ]:
# 6-4. RPM Look-up Table Bar Chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

mat_keys = list(MATERIALS.keys())
size_keys = list(TARGET_SIZES.keys())
x = np.arange(len(mat_keys))
width = 0.25
size_colors = ['#e74c3c', '#f39c12', '#3498db']

# A축 RPM
for i, sk in enumerate(size_keys):
    vals = [RPM_LOOKUP[mk][sk]['A'] for mk in mat_keys]
    bars = ax1.bar(x + i * width, vals, width, label=TARGET_SIZES[sk]['name_kr'],
                   color=size_colors[i], alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
                 str(val), ha='center', va='bottom', fontsize=9, fontweight='bold')

ax1.set_xlabel('Material', fontsize=12)
ax1.set_ylabel('RPM', fontsize=12)
ax1.set_title('A-Axis (Primary) Optimal RPM', fontsize=13, fontweight='bold')
ax1.set_xticks(x + width)
ax1.set_xticklabels([MATERIALS[mk]['name_kr'] for mk in mat_keys])
ax1.legend(title='Target Size')
ax1.set_ylim(0, 1650)

# B축 RPM
for i, sk in enumerate(size_keys):
    vals = [RPM_LOOKUP[mk][sk]['B'] for mk in mat_keys]
    bars = ax2.bar(x + i * width, vals, width, label=TARGET_SIZES[sk]['name_kr'],
                   color=size_colors[i], alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
                 str(val), ha='center', va='bottom', fontsize=9, fontweight='bold')

ax2.set_xlabel('Material', fontsize=12)
ax2.set_ylabel('RPM', fontsize=12)
ax2.set_title('B-Axis (Secondary) Optimal RPM', fontsize=13, fontweight='bold')
ax2.set_xticks(x + width)
ax2.set_xticklabels([MATERIALS[mk]['name_kr'] for mk in mat_keys])
ax2.legend(title='Target Size')
ax2.set_ylim(0, 1650)

plt.suptitle('RPM Look-up Table by Material and Target Size', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 6-4. RPM 룩업 테이블 막대 차트
4종 소재 x 3종 목표 크기에 대한 최적 RPM 기준값을 시각적으로 비교합니다.

In [ ]:
# 6-3. PID Component Decomposition
r = demo_result

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

# P항
axes[0].plot(r['time'], r['p_history'], 'r-', linewidth=1.2, label='P-term (Proportional)')
axes[0].axhline(0, color='gray', linestyle='-', alpha=0.3)
axes[0].set_ylabel('P Output', fontsize=11)
axes[0].legend(loc='upper right')
axes[0].set_title('PID Component Decomposition - Battery Material, Target 30mm', fontsize=14, fontweight='bold')

# I항
axes[1].plot(r['time'], r['i_history'], 'g-', linewidth=1.2, label='I-term (Integral)')
axes[1].axhline(0, color='gray', linestyle='-', alpha=0.3)
axes[1].set_ylabel('I Output', fontsize=11)
axes[1].legend(loc='upper right')

# D항
axes[2].plot(r['time'], r['d_history'], 'm-', linewidth=1.2, label='D-term (Derivative)')
axes[2].axhline(0, color='gray', linestyle='-', alpha=0.3)
axes[2].set_ylabel('D Output', fontsize=11)
axes[2].legend(loc='upper right')

# 총 출력
axes[3].plot(r['time'], r['pid_output'], 'k-', linewidth=1.5, label='Total PID Output')
axes[3].axhline(0, color='gray', linestyle='-', alpha=0.3)
axes[3].set_xlabel('Time Step (min)', fontsize=12)
axes[3].set_ylabel('Total Output', fontsize=11)
axes[3].legend(loc='upper right')

plt.tight_layout()
plt.show()

### 6-3. PID 성분 분해 (P, I, D 항)
PID 제어기의 각 성분(비례, 적분, 미분)이 출력에 미치는 기여도를 분석합니다.

In [ ]:
# 6-2. RPM Adjustment Trajectory
r = demo_result
base_rpm = RPM_LOOKUP['battery']['fine']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# A축
ax1.plot(r['time'], r['rpm_a'], 'r-', linewidth=1.5, label='A-Axis RPM (Actual)')
ax1.axhline(base_rpm['A'], color='r', linestyle='--', alpha=0.5,
            label=f'A-Axis Optimal ({base_rpm["A"]} RPM)')
ax1.set_ylabel('A-Axis RPM', fontsize=12)
ax1.set_title('RPM Adjustment Trajectory - Battery Material, Target 30mm', fontsize=14, fontweight='bold')
ax1.legend(loc='lower right', fontsize=10)
ax1.set_ylim(0, 1600)

# B축
ax2.plot(r['time'], r['rpm_b'], 'b-', linewidth=1.5, label='B-Axis RPM (Actual)')
ax2.axhline(base_rpm['B'], color='b', linestyle='--', alpha=0.5,
            label=f'B-Axis Optimal ({base_rpm["B"]} RPM)')
ax2.set_xlabel('Time Step (min)', fontsize=12)
ax2.set_ylabel('B-Axis RPM', fontsize=12)
ax2.legend(loc='lower right', fontsize=10)
ax2.set_ylim(0, 1600)

plt.tight_layout()
plt.show()

### 6-2. RPM 조정 궤적 (A축/B축)
PID 제어기에 의한 A축, B축 RPM의 시간에 따른 변화를 확인합니다.

In [ ]:
# 6-1. Setpoint Tracking
r = demo_result
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(r['time'], r['size'], 'b-', linewidth=1.5, alpha=0.8, label='Actual Shred Size')
ax.plot(r['time'], r['setpoint'], 'r--', linewidth=2, label='Setpoint (30mm)')

# 정착 시간 표시
tolerance = r['target_size'] * 0.05
ax.axhline(r['target_size'] + tolerance, color='orange', linestyle=':', alpha=0.5, label=f'+/- 5% Tolerance ({tolerance:.1f}mm)')
ax.axhline(r['target_size'] - tolerance, color='orange', linestyle=':', alpha=0.5)

# 정착 시간 영역
if r['settling_time'] < 120:
    ax.axvspan(0, r['settling_time'], alpha=0.08, color='yellow', label=f'Settling Region ({r["settling_time"]} steps)')
    ax.axvline(r['settling_time'], color='green', linestyle='-.', alpha=0.6)

ax.fill_between(r['time'], r['target_size'] - tolerance, r['target_size'] + tolerance,
                alpha=0.1, color='orange')

ax.set_xlabel('Time Step (min)', fontsize=12)
ax.set_ylabel('Shred Size (mm)', fontsize=12)
ax.set_title('PID Setpoint Tracking - Battery Material, Target 30mm (Fine)', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.set_xlim(0, 120)

plt.tight_layout()
plt.show()

## Step 6. 시각화

### 6-1. 목표값 추적 (Setpoint Tracking)
배터리 소재, 30mm 목표에 대한 PID 제어 결과를 시간에 따라 표시합니다.

In [ ]:
np.random.seed(42)

# 전체 시나리오 실행
all_results = {}
for mat_key in MATERIALS:
    for size_key in TARGET_SIZES:
        key = f"{mat_key}_{size_key}"
        all_results[key] = run_pid_simulation(mat_key, size_key, n_steps=120)

# 결과 요약 테이블
print("=" * 75)
print(f"{'소재':<10} {'목표크기':<12} {'정착시간':>8} {'오버슈트':>10} "
      f"{'SS오차':>8} {'최종크기':>8}")
print(f"{'':10} {'':12} {'(스텝)':>8} {'(%)':>10} {'(mm)':>8} {'(mm)':>8}")
print("=" * 75)

for mat_key in MATERIALS:
    for size_key in TARGET_SIZES:
        key = f"{mat_key}_{size_key}"
        r = all_results[key]
        final_size = np.mean(r['size'][-20:])
        print(f"{MATERIALS[mat_key]['name_kr']:<10} "
              f"{TARGET_SIZES[size_key]['name_kr']:<12} "
              f"{r['settling_time']:>8d} "
              f"{r['overshoot_pct']:>10.1f} "
              f"{r['ss_error']:>8.2f} "
              f"{final_size:>8.2f}")

print("=" * 75)
print(f"총 {len(all_results)}개 시나리오 시뮬레이션 완료")

## Step 5. 전체 시나리오 시뮬레이션 (4소재 x 3크기 = 12개)

모든 소재-목표크기 조합에 대해 PID 제어 시뮬레이션을 수행하고 성능 지표를 비교합니다.

In [ ]:
np.random.seed(42)

# 대표 시나리오 실행: 배터리 + 미세(30mm)
demo_result = run_pid_simulation('battery', 'fine', n_steps=120)

print("=== PID 제어 시뮬레이션 결과 (배터리, 30mm) ===")
print(f"  목표 파쇄 크기: {demo_result['target_size']} mm")
print(f"  정착 시간: {demo_result['settling_time']} 스텝 ({demo_result['settling_time']}분)")
print(f"  오버슈트: {demo_result['overshoot_pct']:.1f}%")
print(f"  정상상태 오차: {demo_result['ss_error']:.2f} mm")
print(f"  최종 파쇄 크기 (평균, 마지막 20스텝): {np.mean(demo_result['size'][-20:]):.2f} mm")
print(f"  최종 A축 RPM: {demo_result['rpm_a'][-1]:.0f}")
print(f"  최종 B축 RPM: {demo_result['rpm_b'][-1]:.0f}")

## Step 4. PID 제어 루프 실행 (배터리, 30mm 목표)

대표 시나리오로 **배터리 소재, 목표 파쇄 크기 30mm(미세)**를 먼저 시뮬레이션합니다.
- 120 타임스텝 (2시간, 1분 간격)
- 콜드스타트에서 목표 수렴 과정 관찰

In [ ]:
def shred_size_model(rpm_a, rpm_b, material_key, prev_size=None, dt=1.0):
    """
    파쇄 크기 시뮬레이션 모델
    - rpm_a: A축 RPM (주축)
    - rpm_b: B축 RPM (보조축)
    - material_key: 소재 키
    - prev_size: 이전 시간의 파쇄 크기 (1차 지연 반영)
    - dt: 시간 간격
    반환: 현재 파쇄 크기 (mm)
    """
    mat = MATERIALS[material_key]
    hardness = mat['hardness']
    noise_std = mat['noise_std']
    lag = mat['response_lag']

    # 기본 크기 모델: RPM이 높을수록 크기 감소
    # base_size = 120mm (무부하 상태의 입력 크기)
    base_size = 120.0

    # RPM 효과 (정규화 후 적용)
    rpm_a_effect = 35.0 * (rpm_a / 1000.0)  # A축 기여 (주축, 더 큰 영향)
    rpm_b_effect = 25.0 * (rpm_b / 1000.0)  # B축 기여 (보조축)

    # 경도 보정: 경도가 높을수록 파쇄가 어려움
    hardness_offset = 15.0 * hardness

    # 목표 크기 계산
    target_size = base_size - rpm_a_effect - rpm_b_effect + hardness_offset

    # 공정 노이즈
    noise = np.random.normal(0, noise_std)

    # 1차 지연 응답 (시스템 관성)
    if prev_size is not None:
        tau = lag  # 시정수
        alpha = dt / (tau + dt)
        actual_size = prev_size + alpha * (target_size - prev_size) + noise
    else:
        actual_size = target_size + noise

    # 크기는 양수
    return max(actual_size, 5.0)


def run_pid_simulation(material_key, size_key, n_steps=120):
    """
    PID 제어 시뮬레이션 실행
    반환: 시뮬레이션 결과 딕셔너리
    """
    target_size = TARGET_SIZES[size_key]['size']
    base_rpm = RPM_LOOKUP[material_key][size_key]
    gains = PID_GAINS[material_key]

    # PID 제어기 생성
    pid = PIDController(
        Kp=gains['Kp'], Ki=gains['Ki'], Kd=gains['Kd'],
        output_min=100, output_max=1500
    )

    # 초기 상태: 기준 RPM의 60%에서 시작 (콜드스타트)
    rpm_a = base_rpm['A'] * 0.6
    rpm_b = base_rpm['B'] * 0.6

    # 초기 파쇄 크기 (높은 값에서 시작)
    current_size = 100.0

    # 기록 배열
    time_steps = np.arange(n_steps)
    size_history = np.zeros(n_steps)
    rpm_a_history = np.zeros(n_steps)
    rpm_b_history = np.zeros(n_steps)
    error_history = np.zeros(n_steps)
    setpoint_history = np.full(n_steps, target_size)

    for t in range(n_steps):
        # 현재 파쇄 크기 측정
        current_size = shred_size_model(
            rpm_a, rpm_b, material_key, prev_size=current_size
        )

        # PID 제어 출력 계산
        pid_output = pid.compute(target_size, current_size)

        # RPM 조정 (PID 출력을 RPM 변화량으로 변환)
        rpm_adjustment = pid_output * 0.5

        # A축/B축 RPM 업데이트
        rpm_a = np.clip(base_rpm['A'] * 0.5 + rpm_adjustment,
                        100, 1500)
        rpm_b = np.clip(base_rpm['B'] * 0.5 + rpm_adjustment * 0.9,
                        100, 1500)

        # 기록
        size_history[t] = current_size
        rpm_a_history[t] = rpm_a
        rpm_b_history[t] = rpm_b
        error_history[t] = current_size - target_size

    # 성능 지표 계산
    # 정착 시간: 오차가 ±5% 이내로 들어간 시점
    tolerance = target_size * 0.05
    settling_time = n_steps  # 기본값
    for t in range(n_steps - 1, -1, -1):
        if abs(error_history[t]) > tolerance:
            settling_time = min(t + 1, n_steps)
            break
    else:
        settling_time = 0

    # 오버슈트
    if target_size < 100:  # 크기를 줄이는 방향
        undershoot = target_size - np.min(size_history[5:])
        overshoot_pct = max(0, undershoot / target_size * 100)
    else:
        overshoot_pct = 0.0

    # 정상상태 오차 (마지막 20스텝 평균)
    ss_error = np.mean(np.abs(error_history[-20:]))

    return {
        'time': time_steps,
        'size': size_history,
        'setpoint': setpoint_history,
        'rpm_a': rpm_a_history,
        'rpm_b': rpm_b_history,
        'error': error_history,
        'p_history': np.array(pid.p_history),
        'i_history': np.array(pid.i_history),
        'd_history': np.array(pid.d_history),
        'pid_output': np.array(pid.output_history),
        'settling_time': settling_time,
        'overshoot_pct': overshoot_pct,
        'ss_error': ss_error,
        'material': material_key,
        'size_key': size_key,
        'target_size': target_size,
    }

print("파쇄 크기 시뮬레이션 모델 정의 완료")
print("  - shred_size_model(): 파쇄 크기 물리 모델")
print("  - run_pid_simulation(): PID 제어 루프 실행")

## Step 3. 파쇄 크기 시뮬레이션 모델

파쇄 크기는 RPM, 소재 경도, 공정 노이즈의 함수로 모델링합니다.

**모델 수식:**
```
size(t) = base_size - α × (RPM_A/1000) - β × (RPM_B/1000) + hardness_offset + noise
```
- RPM이 높을수록 파쇄 크기가 작아짐 (역비례 관계)
- 소재 경도가 높을수록 같은 RPM에서 파쇄 크기가 큼
- 1차 지연 응답 (시스템 관성 반영)

In [ ]:
# 소재 특성 정의
MATERIALS = {
    'battery':  {'name_kr': '배터리',   'hardness': 0.70, 'noise_std': 1.5,
                 'response_lag': 3,  'color': '#e74c3c'},
    'plastic':  {'name_kr': '플라스틱', 'hardness': 0.40, 'noise_std': 1.0,
                 'response_lag': 2,  'color': '#3498db'},
    'metal':    {'name_kr': '금속',     'hardness': 1.00, 'noise_std': 2.0,
                 'response_lag': 4,  'color': '#95a5a6'},
    'mixed':    {'name_kr': '혼합',     'hardness': 0.75, 'noise_std': 2.5,
                 'response_lag': 3,  'color': '#f39c12'},
}

# 목표 파쇄 크기 (mm)
TARGET_SIZES = {
    'fine':   {'size': 30, 'name_kr': '미세(30mm)'},
    'medium': {'size': 50, 'name_kr': '중간(50mm)'},
    'coarse': {'size': 80, 'name_kr': '조대(80mm)'},
}

# RPM 룩업 테이블: [A축 RPM, B축 RPM]
# A축: 1차 파쇄 (주축), B축: 2차 파쇄 (보조축)
RPM_LOOKUP = {
    'battery': {
        'fine':   {'A': 1200, 'B': 1100},
        'medium': {'A': 900,  'B': 800},
        'coarse': {'A': 600,  'B': 500},
    },
    'plastic': {
        'fine':   {'A': 1000, 'B': 950},
        'medium': {'A': 750,  'B': 700},
        'coarse': {'A': 500,  'B': 450},
    },
    'metal': {
        'fine':   {'A': 1400, 'B': 1300},
        'medium': {'A': 1100, 'B': 1000},
        'coarse': {'A': 800,  'B': 700},
    },
    'mixed': {
        'fine':   {'A': 1250, 'B': 1150},
        'medium': {'A': 950,  'B': 850},
        'coarse': {'A': 650,  'B': 550},
    },
}

# PID 게인 (소재별 튜닝)
PID_GAINS = {
    'battery':  {'Kp': 15.0, 'Ki': 0.8, 'Kd': 5.0},
    'plastic':  {'Kp': 12.0, 'Ki': 1.0, 'Kd': 3.0},
    'metal':    {'Kp': 20.0, 'Ki': 0.6, 'Kd': 8.0},
    'mixed':    {'Kp': 18.0, 'Ki': 0.7, 'Kd': 6.0},
}

print("RPM 룩업 테이블 정의 완료")
print(f"  - 소재: {len(MATERIALS)}종")
print(f"  - 목표 크기: {len(TARGET_SIZES)}종")
print(f"  - 총 시나리오: {len(MATERIALS) * len(TARGET_SIZES)}개")
print()

# 룩업 테이블 출력
print(f"{'소재':<10} {'목표크기':<12} {'A축 RPM':>8} {'B축 RPM':>8}")
print("-" * 42)
for mat_key, mat_info in MATERIALS.items():
    for size_key, size_info in TARGET_SIZES.items():
        rpm = RPM_LOOKUP[mat_key][size_key]
        print(f"{mat_info['name_kr']:<10} {size_info['name_kr']:<12} "
              f"{rpm['A']:>8} {rpm['B']:>8}")

## Step 2. RPM 룩업 테이블

소재별/목표 크기별 최적 RPM 기준값을 정의합니다.

| 소재 | 경도 계수 | 특성 |
|------|-----------|------|
| 배터리 | 0.7 | 중간 경도, 내부 전해질 주의 |
| 플라스틱 | 0.4 | 낮은 경도, 빠른 파쇄 |
| 금속 | 1.0 | 높은 경도, 높은 RPM 필요 |
| 혼합 | 0.75 | 복합 소재, 변동성 큼 |

In [ ]:
class PIDController:
    """
    산업용 PID 제어기 (Anti-windup 포함)
    - Kp, Ki, Kd: PID 게인
    - output_min, output_max: 출력 제한 (RPM 범위)
    - integral_limit: 적분항 제한 (Anti-windup)
    """
    def __init__(self, Kp, Ki, Kd, output_min=100, output_max=1500,
                 integral_limit=500):
        self.Kp = Kp
        self.Ki = Ki
        self.Kd = Kd
        self.output_min = output_min
        self.output_max = output_max
        self.integral_limit = integral_limit

        # 내부 상태
        self.integral = 0.0
        self.prev_error = 0.0
        self.prev_output = 0.0

        # 기록용
        self.p_history = []
        self.i_history = []
        self.d_history = []
        self.output_history = []

    def reset(self):
        """제어기 상태 초기화"""
        self.integral = 0.0
        self.prev_error = 0.0
        self.prev_output = 0.0
        self.p_history = []
        self.i_history = []
        self.d_history = []
        self.output_history = []

    def compute(self, setpoint, measured, dt=1.0):
        """
        PID 출력 계산
        - setpoint: 목표값 (목표 파쇄 크기)
        - measured: 현재 측정값 (현재 파쇄 크기)
        - dt: 시간 간격
        반환: RPM 조정값
        """
        # 오차 계산 (크기가 크면 RPM을 올려야 하므로 부호 반전)
        error = measured - setpoint

        # P항
        p_term = self.Kp * error

        # I항 (Anti-windup 적용)
        self.integral += error * dt
        self.integral = np.clip(self.integral,
                                -self.integral_limit,
                                self.integral_limit)
        i_term = self.Ki * self.integral

        # D항
        derivative = (error - self.prev_error) / dt
        d_term = self.Kd * derivative

        # 총 출력
        output = p_term + i_term + d_term

        # 출력 제한
        output = np.clip(output, self.output_min, self.output_max)

        # 기록 저장
        self.p_history.append(p_term)
        self.i_history.append(i_term)
        self.d_history.append(d_term)
        self.output_history.append(output)

        # 상태 업데이트
        self.prev_error = error
        self.prev_output = output

        return output

print("PID 제어기 클래스 정의 완료")
print(f"  - Anti-windup 적분 제한: ±500")
print(f"  - 출력 범위: 100 ~ 1500 RPM")

## Step 1. PID 제어기 클래스 정의

PID 제어기의 핵심 요소:
- **P (비례):** 현재 오차에 비례하여 출력 조정
- **I (적분):** 누적 오차를 보정하여 정상상태 오차 제거
- **D (미분):** 오차 변화율을 감지하여 오버슈트 억제
- **Anti-windup:** 적분항의 과도한 축적 방지 (출력 포화 시)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# 시각화 설정
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

np.random.seed(42)
print("라이브러리 로드 완료")

## Step 0. 라이브러리 임포트

# 🏭 PID 제어 기반 파쇄기 RPM 최적화

**목적:** 파쇄기의 A축/B축 RPM을 PID 제어기로 자동 조정하여 목표 파쇄 크기를 정밀하게 달성

**주요 내용:**
- PID 제어기 설계 (비례-적분-미분) + Anti-windup
- 4종 소재 × 3종 목표 크기 = 12개 시나리오 시뮬레이션
- 120 타임스텝 (2시간, 1분 간격) 생산 레벨 시뮬레이션
- RPM 룩업 테이블 및 최적 파라미터 도출

**소재:** 배터리, 플라스틱, 금속, 혼합  
**목표 파쇄 크기:** 30mm (미세), 50mm (중간), 80mm (조대)